In [1]:
import pandas as pd
import numpy as np

import bz2
import _pickle as cPickle
from pathlib import Path

In [2]:
def save_compressed_pickle(filename, data):
    with bz2.BZ2File(filename + '.pbz2', 'w') as f:
        cPickle.dump(data, f)
def load_compressed_pickle(filename):
    data = bz2.BZ2File(filename + '.pbz2', 'rb')
    data = cPickle.load(data)
    return data

In [3]:
repo_folder = Path('../..')
data_folder  = repo_folder / 'data'/ 'this_project' / '6_transporterKO'
metabolomics_folder = data_folder / 'D_big_screen'


In [4]:
median_fn = metabolomics_folder / 'G_median_z_scores_TIC_norm_keio.csv'
df_data = pd.read_csv(median_fn)
df_data.rename(columns={'Z-score median': 'Median Z-score'}, inplace=True)

ionMz_annotation_fn =  metabolomics_folder / 'H_ionMz_annotation.csv'
df_ionMz = pd.read_csv(ionMz_annotation_fn)

sample_metadata_fn = metabolomics_folder /  'I_sample_metadata_keio.csv'
df_sample_metadata = pd.read_csv(sample_metadata_fn, index_col=0)

In [5]:
df_w_mz = df_data.merge(df_ionMz, on='ionMz', how='left')
df = df_w_mz.merge(df_sample_metadata, on=['Batch-Tube', 'Timepoint'], how='left')

In [6]:
xval = 'Hours'
yval = 'Median Z-score'
ystd  = 'Z-score std'

In [7]:
all_metabolites = np.unique(df["Metabolite"]).astype(str)
all_strains = np.unique(df["Strain"]).astype(str)
non_wt_strains = np.array([s for s in all_strains if s != 'WT'])

In [8]:
all_metabolites

array(['(-)-Ureidoglycolate', '(2-Naphthyl)methanol',
       '(6Z,9Z,12Z)-Octadecatrienoic acid', '(9Z)-Hexadecenoic acid',
       '(E)-3-(Methoxycarbonyl)pent-2-enedioate', '(Iso)Citrate',
       '(Iso)Leucine', '(R) 2,3-Dihydroxy-3-methylvalerate',
       '(R)-2-Hydroxybutane-1,2,4-tricarboxylate',
       "(R)-4'-Phosphopantothenoyl-L-cysteine",
       '1,4-Dihydroxy-2-naphthoate', '1,6-Anhydro-N-acetyl-beta-muramate',
       '1-(5-Phospho-D-ribosyl)-5-amino-4-imidazolecarboxylate',
       '1-Nitronaphthalene-5,6-oxide', '1-Palmitoylglycerol 3-phosphate',
       '1D-myo-Inositol 1,3,4,5,6-pentakisphosphate', "2',3'-Cyclic UMP",
       '2(alpha-D-Mannosyl)-D-glycerate',
       '2,3,4,5-Tetrahydrodipicolinate', '2,3-Diaminopropanoate',
       '2-(3,4-dihydroxybenzoyloxy)-4,6-dihydroxybenzoate',
       '2-Aceto-2-hydroxybutanoate',
       '2-Amino-3-oxo-4-phosphonooxybutyrate',
       '2-Amino-4-hydroxy-6-hydroxymethyl-7,8-dihydropteridine',
       '2-Aminobenzoic acid', '2-Aminobut-2-e

In [9]:
save_compressed_pickle("processed_data/keio_all_metabolites_list", all_metabolites)
save_compressed_pickle("processed_data/keio_non_wt_strains_list", non_wt_strains)

In [10]:
data_dict = {cur_strain: {cur_metabolite: None for cur_metabolite in all_metabolites} for cur_strain in non_wt_strains}
spline_dict = {cur_strain: {cur_metabolite: None for cur_metabolite in all_metabolites} for cur_strain in non_wt_strains}

In [11]:
for cur_strain in non_wt_strains:
    strains = [cur_strain, 'WT']

    batch = df.loc[df.Strain==strains[0], 'Batch'].values[0]
    cur_pair_metabolites = np.unique(df[df.Strain.isin(strains)&(df.Batch.isin([batch]))]["Metabolite"])
    for cur_metabolite in cur_pair_metabolites:
        idx =  df.Strain.isin(strains)&(df.Metabolite==cur_metabolite)&(df.Batch.isin([batch]))

        if(len(np.where(idx)[0])==0):
            print(f'Skipping {cur_strain} - {cur_metabolite} due to no data')
            continue

        # Get the data for each strain
        data = df.loc[idx].copy()

        # Make sure there are no duplicated x-values at all
        is_duplicated = data[xval].duplicated()
        if np.any(is_duplicated):
            # Add tiny random noise to the x-values to break ties
            data.loc[is_duplicated, xval] += np.random.normal(0, 1e-3, size=is_duplicated.sum())

        data.sort_values(by = xval, inplace=True)

        # Separate by strain
        strain1, strain2 = strains
        x1 = data.loc[data['Strain'] == strain1, xval].values
        y1 = data.loc[data['Strain'] == strain1, yval].values
        y1_std= data.loc[data['Strain'] == strain1, ystd].values
        x2 = data.loc[data['Strain'] == strain2, xval].values
        y2 = data.loc[data['Strain'] == strain2, yval].values
        y2_std= data.loc[data['Strain'] == strain2, ystd].values

        data_dict[cur_strain][cur_metabolite] = [[x1,y1,y1_std], [x2,y2,y2_std]]
    print(f'Completed strain: {cur_strain}')

Completed strain: acrB
Completed strain: actP
Completed strain: adeP
Completed strain: ansP


KeyboardInterrupt: 

In [ ]:
save_compressed_pickle("processed_data/spline_dict_keio_all_strains", spline_dict)
save_compressed_pickle("processed_data/data_dict_keio_all_strains_with_std", data_dict)